# Vitalis: Clinical Time-Series Forecasting & Depletion Guardrail Classifier
## Academic Course Alignment: BCSE335L (Healthcare Analytics)
- **Module 3: Machine Learning Foundations** (Feature Engineering, Imputation, Tree Ensembles)
- **Module 4: Measuring Healthcare Quality** (Medicare HVBP / HAC Composite Penalties)
- **Module 5: Predictive Models in Healthcare** (Time-Series Resampling, Meta Prophet Forecasting, XGBoost Stockout Guardrails)

### Abstract
This notebook demonstrates the end-to-end data processing and model development pipeline for **Vitalis**, an agentic clinical logistics platform. We formulate two complementary predictive models:
1. **Model A (Meta Prophet)**: Rolling 14-day network consumption demand forecaster.
2. **Model B (XGBoost Classifier)**: 72-hour donor facility stockout risk estimator enforcing clinical safety guardrails.

In [1]:
import os
import math
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from prophet import Prophet
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, recall_score, classification_report

# Plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All dependencies successfully imported.')

## 1. Dataset Ingestion & NHAMCS Extraction
We model emergency department transfusion volume and Emergency Severity Index (ESI) acuity on the CDC NHAMCS emergency department surveillance survey.

In [2]:
# Load preprocessed NHAMCS consumption records
data_path = '../data/nhamcs_consumption_history.csv'
df = pd.read_csv(data_path, parse_dates=['date'])
print(f'Total records ingested: {len(df):,}')
df.head()

## 2. Feature Engineering & Cyclical Encodings
Emergency admissions exhibit strong diurnal and seasonal periodicity:
$$\text{day}_{\sin} = \sin\left(\frac{2\pi \cdot \text{day}}{7}\right), \quad \text{day}_{\cos} = \cos\left(\frac{2\pi \cdot \text{day}}{7}\right)$$
$$\text{month}_{\sin} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{month}_{\cos} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$

In [3]:
# Engineer cyclical time features & 7-day rolling statistics
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['sin_day'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['cos_day'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

# 7-day rolling burn rate per facility
df = df.sort_values(['facility_id', 'date']).reset_index(drop=True)
df['burn_rate_7d'] = df.groupby('facility_id')['units_consumed'].transform(
    lambda s: s.rolling(7, min_periods=1).mean()
)
df[['facility_name', 'date', 'units_consumed', 'sin_day', 'cos_day', 'burn_rate_7d']].tail()

## 3. Model A: Meta Prophet Forecasting Engine
Predicts daily demand for Packed Red Blood Cells (PRBC) over a rolling 14-day horizon.
- Additive decomposition: $y(t) = g(t) + s(t) + h(t) + \epsilon_t$
- 80% confidence uncertainty intervals ($[\hat{y}_{\text{lower}}, \hat{y}_{\text{upper}}]$)
- Evaluated against Mean Absolute Scaled Error (MASE < 1.15)

In [4]:
daily_agg = df.groupby('date')['units_consumed'].sum().reset_index()
daily_agg.columns = ['ds', 'y']

split_idx = int(len(daily_agg) * 0.85)
train_df = daily_agg.iloc[:split_idx]
test_df = daily_agg.iloc[split_idx:]

prophet_model = Prophet(
    growth='linear',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.80
)
prophet_model.fit(train_df)

# Generate 14-day forecast
future = prophet_model.make_future_dataframe(periods=14, freq='D')
forecast = prophet_model.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(14)

## 4. Model B: XGBoost Depletion Guardrail Classifier
Predicts 72-hour donor depletion risk $P(Y = 1 \mid \vec{x})$:
$$\vec{x} = [\text{current\_inventory} - k, \hat{y}_{t \le 3}, \text{HAC\_score}, \text{trauma\_level}, \text{burn\_rate}_{7d}]$$
Target: Area Under Precision-Recall Curve (PR-AUC $\ge 0.82$) and Recall $\ge 0.90$.

In [5]:
# Load trained XGBoost model and inspect metadata
with open('../models/xgb_metadata.json') as f:
    meta = json.load(f)
print('XGBoost Model Performance Metadata:')
for k, v in meta.items():
    print(f'  {k}: {v}')

clf = xgb.XGBClassifier()
clf.load_model('../models/xgb_stockout.json')
print('\nModel loaded successfully from ../models/xgb_stockout.json')

## 5. Clinical Safety Guardrail Verification
We evaluate an illustrative scenario where an emergency request draws $k = 4$ units from candidate facilities.
Any facility where $\text{Risk}_{\text{stockout}} > 0.40$ is strictly disqualified from being an outbound donor.

In [6]:
test_candidates = [
    {'name': 'RBTC Central', 'current_inv': 28, 'k': 4, 'yhat_3d': 22.0, 'hac': 98.40, 'trauma': 1, 'burn_rate': 6.5},
    {'name': 'Fortis Malar', 'current_inv': 4, 'k': 4, 'yhat_3d': 18.0, 'hac': 74.00, 'trauma': 2, 'burn_rate': 4.2},
    {'name': 'RGGGH Apex', 'current_inv': 14, 'k': 4, 'yhat_3d': 20.0, 'hac': 91.20, 'trauma': 1, 'burn_rate': 5.8},
]

print(f'{"Candidate Facility":<25} | {"Remaining":<10} | {"Risk Score":<12} | {"Guardrail Status":<15}')
print('-' * 70)
for c in test_candidates:
    feat = np.array([[c['current_inv'] - c['k'], c['yhat_3d'], c['hac'], c['trauma'], c['burn_rate']]])
    risk = float(clf.predict_proba(feat)[0, 1])
    status = 'DISQUALIFIED (>40%)' if risk > 0.40 else 'APPROVED (<=40%)'
    print(f"{c['name']:<25} | {c['current_inv'] - c['k']:<10} | {risk:.3f}        | {status:<15}")